In [ ]:
import numpy as np 
import sympy as sp 
import pandas as pd
import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

Helpers

In [ ]:
# rewritten optimal version:
class RandomWalk:
    def __init__(self, p: float , domain: list[int, int]):
        self.p = p
        self.domain = domain

    def simulate_absorbing(self, x_0: int, n_trials: int, max_steps: int):
        # initialize on x_0
        trajectories = np.zeros((n_trials, max_steps))
        trajectories[:, 0] = x_0

        # next steps 
        for t in range(1, max_steps):
            # params 
            ## current positions 
            current_position = trajectories[:, t-1]

            # New moves 
            ## we choose at random new steps 
            moves = np.random.choice([-1, 1], size=n_trials, p=[1-self.p, self.p])
            next_moves = current_position + moves 

            # Barrier
            ## create logic mask
            is_absorbed = (current_position <= self.domain[0]) | (current_position >= self.domain[1])
            ## if all states are absorbed 
            if np.all(is_absorbed):
                trajectories[:, t:] = current_position[:, np.newaxis]
                break
            
            # update trajectories trajectories 
            trajectories[:, t] = np.where(is_absorbed, current_position, next_moves)

            return trajectories

    def simulate_elasting(self, x_0: int, n_trials: int, max_steps: int):
        # initialize on x_0
        trajectories = np.zeros((n_trials, max_steps))
        trajectories[:, 0] = x_0

        # next steps 
        for t in range(1, max_steps):
            # params 
            ## current positions 
            current_position = trajectories[:, t-1]

            # New moves 
            ## we choose at random flags (0 with 1-p, 1 with p )
            moves = np.random.choice([0, 1], size=n_trials, p=[1 - self.p, self.p])
            ## we create flag 
            is_min = moves == 1 
            ## update trajectories trajectories 
            trajectories[is_min: t] = np.min(current_position[is_min] + 1, np.ones([is_min]) * self.domain[1])
            trajectories[~is_min: t] = np.max(current_position[is_min.sum()] - 1, np.ones(~is_min.sum()) *  self.domain[0])
        
        return trajectories

In [ ]:
random_walk = RandomWalk(0.4, domain=[0, 20])
random_walk.simulate_elasting(3, 40, 20)

In [ ]:
class RandomWalk():
    def __init__(self, p: float, domain: list[int], n : int=0, x_0: int=None):
        self.p = p # probability of going upwards 
        self.n = n # number of iterations 
        self.domain = domain # domain of random walk
        if x_0 is None:
            x_0 = int(np.mean(domain))
        else:
            self.x_0 = x_0 # init value 
        pass

    def _simulate_single_absorbing_barrier(self):
        
        # params
        dist = stats.bernoulli(self.p)

        ## initialize array
        rtn_val = np.zeros(shape=self.n, dtype=np.int64)
        
        ## init value
        rtn_val[0] =  self.x_0

        # simulate trajectory
        for i in range(1, self.n):
            x_recent = rtn_val[i-1]
            ## check if we hit barrier
            if x_recent in self.domain:
                ### if hit all next values stay at barrier 
                rtn_val[i:self.n] = x_recent
                break
            ## if not hit b
            x = dist.rvs()
            x_change = x if x else -1
            rtn_val[i] = x_recent + x_change


        return rtn_val

    def _simulate_single_elasting_barrier(self):
        
        # params 
        dist = stats.bernoulli(self.p)

        ## initialize array
        rtn_val = np.zeros(shape=self.n, dtype=np.int64)
        
        ## init value
        rtn_val[0] =  self.x_0

        # trajectory
        for i in range(1, self.n):
            x_recent = rtn_val[i-1]
            ## check if we hit barrier
            if x_recent in self.domain:
                ### sample value 
                x = dist.rvs()
                ### apply changes
                if x:
                    # min(x+1, right_barrier)
                    val = np.min([x_recent + 1, self.domain[1]])
                else: 
                    # max(x-1, 0) 
                    val = np.max([x_recent - 1, 0])

                ### proceed to next step
                continue
            ## if not hit b
            ### random sample
            x = dist.rvs()
            ### calculate change
            x_change = x if x else -1
            ### update
            rtn_val[i] = x_recent + x_change


        return rtn_val
    
    def simulate_absorbing_barrier(self, n):
        
        rtn_val = self._simulate_single_absorbing_barrier().reshape(1, -1)

        for _ in range(1, n):
            new_val = self._simulate_single_absorbing_barrier().reshape(1, -1)
            rtn_val = np.concat([rtn_val, new_val], axis=0)
        return rtn_val

    def simulate_elasting_barrier(self, n):
        rtn_val = self._simulate_single_elasting_barrier().reshape(1, -1)

        for _ in range(1, n):
            new_val = self._simulate_single_elasting_barrier().reshape(1, -1)
            rtn_val = np.concat([rtn_val, new_val], axis=0)
        return rtn_val

Random walks with absorbing barriers

In [ ]:
# generate probe
random_walk = RandomWalk(
    p=0.4, 
    domain=[0, 20],
    n=200,
    x_0=10    
)
random_walk.simulate_absorbing_barrier(4).shape

In [ ]:
# probability of hitting barrier (X_n= 0, and X_k) given x_0 as function of x_0  
# we need to change values x_0 and for later we will obtain were on last value is 0 or k, given 

final_results = []

for x_0 in range(0, 21): # i took barriers values so we do not need to add it manually (~ computational cost)
    # probes per init value 
    random_walk.x_0 = x_0
    final_results.append(random_walk.simulate_absorbing_barrier(n=100))


Random walks with elastic barrier

In [ ]:
random_walk = RandomWalk(
    p=0.4, 
    domain=[0, 20],
    n=100,
    x_0=10    
)
result_elastic = random_walk.simulate_elasting_barrier(10000)


In [ ]:
# take last element 
result = result_elastic[:, -1] # take only last values 

# count types nad normalize
uniques_count = np.unique_counts(result)



In [ ]:
# stationary distribution